# 12 — Reducción de dimensionalidad: visualización de *digits* en 2D y 3D

En este notebook comparamos varias técnicas de reducción de dimensionalidad sobre el dataset `digits` de scikit-learn.

## Técnicas

- **PCA (Principal Component Analysis)** — referencia lineal no supervisada.
- **MDS (Multidimensional Scaling)** — intenta preservar distancias entre instancias.
- **Isomap** — construye un grafo de vecinos y preserva distancias geodésicas.
- **t-SNE** — preserva principalmente vecindarios locales; se usa sobre todo para visualización.
- **LDA (Linear Discriminant Analysis)** — técnica supervisada que busca direcciones que separen las clases.
- **UMAP (Uniform Manifold Approximation and Projection)** — reducción no lineal orientada a preservar estructura local y parte de la global.

Cada técnica se visualizará en **2 dimensiones** y en **3 dimensiones**.

> Nota: t-SNE y UMAP son excelentes para visualizar, pero la apariencia de grupos en 2D/3D no debe interpretarse por sí sola como evidencia de clusters reales.


## 1. Instalación e importaciones

`umap-learn` no siempre viene instalado en Colab, por eso se instala explícitamente.


In [ ]:
%pip install -q umap-learn


In [ ]:
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
from sklearn.manifold import MDS, Isomap, TSNE
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import umap.umap_ as umap

warnings.filterwarnings("ignore")

RANDOM_STATE = 42


## 2. Dataset: handwritten digits

Cada observación es una imagen de **8 × 8 píxeles**, por lo que originalmente cada dígito vive en un espacio de **64 dimensiones**.

Usaremos una muestra estratificada para que las técnicas más costosas —especialmente MDS— se puedan ejecutar con comodidad. Cambia `N_SAMPLES` si quieres utilizar más puntos.


In [ ]:
digits = load_digits()

X = digits.data
y = digits.target

print("Forma original de X:", X.shape)
print("Número de clases:", len(np.unique(y)))
print("Clases:", np.unique(y))


In [ ]:
N_SAMPLES = 1200

if N_SAMPLES < len(X):
    X_sample, _, y_sample, _ = train_test_split(
        X,
        y,
        train_size=N_SAMPLES,
        stratify=y,
        random_state=RANDOM_STATE
    )
else:
    X_sample, y_sample = X.copy(), y.copy()

print("Muestra utilizada:", X_sample.shape)


### Ejemplos de los datos originales


In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(10, 4))

for digit, ax in enumerate(axes.ravel()):
    idx = np.where(y == digit)[0][0]
    ax.imshow(digits.images[idx], cmap="gray")
    ax.set_title(f"Dígito {digit}")
    ax.axis("off")

plt.tight_layout()
plt.show()


## 3. Estandarización

Las técnicas basadas en distancias pueden verse afectadas por la escala. En `digits` todos los píxeles ya están en el mismo rango, pero estandarizamos las variables para construir una comparación homogénea.


In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_sample)

print("Media aproximada:", np.mean(X_scaled))
print("Desviación estándar aproximada:", np.std(X_scaled))


## 4. Funciones auxiliares de visualización


In [ ]:
def plot_embedding_2d(Z, labels, title):
    """Visualiza una representación de dos dimensiones."""
    plt.figure(figsize=(9, 7))
    scatter = plt.scatter(
        Z[:, 0],
        Z[:, 1],
        c=labels,
        cmap="tab10",
        s=18,
        alpha=0.8
    )
    plt.xlabel("Dimensión 1")
    plt.ylabel("Dimensión 2")
    plt.title(title)
    plt.colorbar(scatter, ticks=range(10), label="Dígito")
    plt.grid(alpha=0.2)
    plt.show()


def plot_embedding_3d(Z, labels, title):
    """Visualiza una representación de tres dimensiones."""
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection="3d")

    scatter = ax.scatter(
        Z[:, 0],
        Z[:, 1],
        Z[:, 2],
        c=labels,
        cmap="tab10",
        s=15,
        alpha=0.75
    )

    ax.set_xlabel("Dimensión 1")
    ax.set_ylabel("Dimensión 2")
    ax.set_zlabel("Dimensión 3")
    ax.set_title(title)

    fig.colorbar(scatter, ax=ax, ticks=range(10), label="Dígito", shrink=0.7)
    plt.show()


tiempos = {}


def medir(nombre, funcion):
    """Ejecuta una reducción y registra su tiempo de cómputo."""
    inicio = time.perf_counter()
    resultado = funcion()
    tiempos[nombre] = time.perf_counter() - inicio
    return resultado


# 5. PCA — Principal Component Analysis

PCA es una técnica **lineal y no supervisada**. Busca nuevas direcciones ortogonales que capturen la mayor varianza posible.

Si (X \in \mathbb{R}^{n \times d}), PCA encuentra ejes sobre los cuales proyectar los datos:

[
Z = XW,
]

donde las columnas de (W) son los autovectores asociados a las mayores varianzas.


In [ ]:
pca_2d = PCA(n_components=2, random_state=RANDOM_STATE)
Z_pca_2d = medir("PCA 2D", lambda: pca_2d.fit_transform(X_scaled))

print("Varianza explicada total (2D):", pca_2d.explained_variance_ratio_.sum())
plot_embedding_2d(Z_pca_2d, y_sample, "PCA — 2D")


In [ ]:
pca_3d = PCA(n_components=3, random_state=RANDOM_STATE)
Z_pca_3d = medir("PCA 3D", lambda: pca_3d.fit_transform(X_scaled))

print("Varianza explicada total (3D):", pca_3d.explained_variance_ratio_.sum())
plot_embedding_3d(Z_pca_3d, y_sample, "PCA — 3D")


# 6. MDS — Multidimensional Scaling

MDS busca una representación de menor dimensión cuyas **distancias entre pares de puntos** sean similares a las distancias originales.

Una forma simplificada del objetivo es minimizar el *stress*:

[
\text{Stress}(Z)
=
\sum_{i<j}
\left(
d_{ij}^{(X)} - d_{ij}^{(Z)}
\right)^2.
]

Por comparar muchos pares de puntos, MDS puede ser considerablemente más costoso que PCA.


In [ ]:
mds_2d = MDS(
    n_components=2,
    random_state=RANDOM_STATE,
    n_init=1,
    max_iter=300,
    n_jobs=-1
)

Z_mds_2d = medir("MDS 2D", lambda: mds_2d.fit_transform(X_scaled))
plot_embedding_2d(Z_mds_2d, y_sample, "MDS — 2D")


In [ ]:
mds_3d = MDS(
    n_components=3,
    random_state=RANDOM_STATE,
    n_init=1,
    max_iter=300,
    n_jobs=-1
)

Z_mds_3d = medir("MDS 3D", lambda: mds_3d.fit_transform(X_scaled))
plot_embedding_3d(Z_mds_3d, y_sample, "MDS — 3D")


# 7. Isomap

Isomap es una técnica no lineal basada en un **grafo de vecinos**.

1. Conecta cada observación con sus vecinos más cercanos.
2. Aproxima la distancia geodésica entre puntos usando caminos mínimos en el grafo.
3. Busca una representación de baja dimensión que preserve esas distancias geodésicas.

La idea es útil cuando los datos se encuentran cerca de una **variedad** (*manifold*) no lineal.


In [ ]:
N_NEIGHBORS = 15

isomap_2d = Isomap(
    n_neighbors=N_NEIGHBORS,
    n_components=2,
    n_jobs=-1
)

Z_isomap_2d = medir("Isomap 2D", lambda: isomap_2d.fit_transform(X_scaled))
plot_embedding_2d(Z_isomap_2d, y_sample, "Isomap — 2D")


In [ ]:
isomap_3d = Isomap(
    n_neighbors=N_NEIGHBORS,
    n_components=3,
    n_jobs=-1
)

Z_isomap_3d = medir("Isomap 3D", lambda: isomap_3d.fit_transform(X_scaled))
plot_embedding_3d(Z_isomap_3d, y_sample, "Isomap — 3D")


# 8. t-SNE — t-distributed Stochastic Neighbor Embedding

t-SNE intenta conservar **vecindarios locales**: puntos similares en alta dimensión reciben una alta probabilidad de permanecer cerca en la proyección.

Su objetivo se basa en minimizar una divergencia entre distribuciones de similitud:

[
KL(P\|Q)
=
\sum_{i \neq j}
p_{ij}
\log
\frac{p_{ij}}{q_{ij}}.
]

La distancia entre grupos separados en una gráfica t-SNE no debe interpretarse literalmente como una distancia global.


In [ ]:
tsne_2d = TSNE(
    n_components=2,
    perplexity=30,
    init="pca",
    learning_rate="auto",
    max_iter=1000,
    random_state=RANDOM_STATE
)

Z_tsne_2d = medir("t-SNE 2D", lambda: tsne_2d.fit_transform(X_scaled))
plot_embedding_2d(Z_tsne_2d, y_sample, "t-SNE — 2D")


In [ ]:
tsne_3d = TSNE(
    n_components=3,
    perplexity=30,
    init="pca",
    learning_rate="auto",
    max_iter=1000,
    random_state=RANDOM_STATE
)

Z_tsne_3d = medir("t-SNE 3D", lambda: tsne_3d.fit_transform(X_scaled))
plot_embedding_3d(Z_tsne_3d, y_sample, "t-SNE — 3D")


# 9. LDA — Linear Discriminant Analysis

LDA es diferente a las técnicas anteriores porque es **supervisada**: utiliza las etiquetas (y).

Busca direcciones que maximicen la separación entre clases respecto a la dispersión dentro de cada clase. Una formulación común es:

[
W^*
=
\arg\max_W
\frac{|W^T S_B W|}
{|W^T S_W W|},
]

donde:

- (S_B): dispersión **entre clases**.
- (S_W): dispersión **dentro de las clases**.

Con (C) clases, LDA puede producir como máximo (C-1) componentes. En `digits`, (C=10), así que 2D y 3D son posibles.


In [ ]:
lda_2d = LinearDiscriminantAnalysis(n_components=2)

Z_lda_2d = medir(
    "LDA 2D",
    lambda: lda_2d.fit_transform(X_scaled, y_sample)
)

plot_embedding_2d(Z_lda_2d, y_sample, "LDA — 2D")


In [ ]:
lda_3d = LinearDiscriminantAnalysis(n_components=3)

Z_lda_3d = medir(
    "LDA 3D",
    lambda: lda_3d.fit_transform(X_scaled, y_sample)
)

plot_embedding_3d(Z_lda_3d, y_sample, "LDA — 3D")


# 10. UMAP — Uniform Manifold Approximation and Projection

UMAP también construye relaciones de vecindad, pero utiliza una representación basada en un **grafo difuso** (*fuzzy simplicial set*) y busca una proyección de baja dimensión que conserve esas relaciones.

Dos hiperparámetros importantes:

- `n_neighbors`: controla cuánto contexto local se usa.
- `min_dist`: controla qué tan compactos pueden quedar los puntos en la proyección.

UMAP suele utilizarse para visualización 2D/3D y puede ser más rápido que t-SNE en conjuntos grandes.


In [ ]:
umap_2d = umap.UMAP(
    n_components=2,
    n_neighbors=15,
    min_dist=0.1,
    random_state=RANDOM_STATE
)

Z_umap_2d = medir("UMAP 2D", lambda: umap_2d.fit_transform(X_scaled))
plot_embedding_2d(Z_umap_2d, y_sample, "UMAP — 2D")


In [ ]:
umap_3d = umap.UMAP(
    n_components=3,
    n_neighbors=15,
    min_dist=0.1,
    random_state=RANDOM_STATE
)

Z_umap_3d = medir("UMAP 3D", lambda: umap_3d.fit_transform(X_scaled))
plot_embedding_3d(Z_umap_3d, y_sample, "UMAP — 3D")


# 11. Comparación visual conjunta en 2D

Esta cuadrícula utiliza exactamente los mismos puntos y etiquetas para facilitar la comparación visual.


In [ ]:
embeddings_2d = {
    "PCA": Z_pca_2d,
    "MDS": Z_mds_2d,
    "Isomap": Z_isomap_2d,
    "t-SNE": Z_tsne_2d,
    "LDA": Z_lda_2d,
    "UMAP": Z_umap_2d,
}

fig, axes = plt.subplots(2, 3, figsize=(18, 11))

for ax, (nombre, Z) in zip(axes.ravel(), embeddings_2d.items()):
    scatter = ax.scatter(
        Z[:, 0],
        Z[:, 1],
        c=y_sample,
        cmap="tab10",
        s=10,
        alpha=0.75
    )
    ax.set_title(nombre)
    ax.set_xlabel("Dimensión 1")
    ax.set_ylabel("Dimensión 2")
    ax.grid(alpha=0.15)

fig.colorbar(
    scatter,
    ax=axes.ravel().tolist(),
    ticks=range(10),
    label="Dígito",
    shrink=0.75
)

plt.suptitle("Comparación de técnicas de reducción de dimensionalidad — 2D", fontsize=16)
plt.show()


# 12. Comparación de tiempos

Los tiempos dependen del hardware y de la versión de las librerías, por lo que esta tabla debe interpretarse como una comparación experimental del entorno donde ejecutes el notebook.


In [ ]:
tabla_tiempos = (
    pd.DataFrame(
        {"Método": list(tiempos.keys()), "Tiempo (s)": list(tiempos.values())}
    )
    .sort_values("Tiempo (s)")
    .reset_index(drop=True)
)

tabla_tiempos


# 13. Resumen conceptual

| Técnica | Tipo | ¿Usa etiquetas? | Qué intenta preservar | Uso típico |
|---|---|---:|---|---|
| PCA | Lineal | No | Varianza global | Compresión, preprocesamiento, visualización |
| MDS | No lineal/general | No | Distancias entre pares | Visualización de relaciones de distancia |
| Isomap | No lineal | No | Distancias geodésicas sobre un grafo | Variedades no lineales |
| t-SNE | No lineal | No | Vecindarios locales | Visualización 2D/3D |
| LDA | Lineal | **Sí** | Separación entre clases | Reducción supervisada antes de clasificar |
| UMAP | No lineal | No | Estructura local y parte de la global | Visualización 2D/3D |

## Preguntas para interpretar los resultados

1. ¿Qué técnica produce la separación visual más clara entre dígitos?
2. ¿Agregar una tercera dimensión cambia sustancialmente la estructura observada?
3. ¿Por qué LDA suele separar bien las clases cuando ya conoce las etiquetas?
4. ¿Qué diferencias aparecen entre t-SNE y UMAP?
5. ¿Qué técnicas parecen preservar estructura global y cuáles enfatizan estructura local?
6. ¿Qué método tarda más en tu ejecución?


## 14. Idea central

Reducir de 64 dimensiones a 2 o 3 significa construir una transformación

[
f:\mathbb{R}^{64}\rightarrow\mathbb{R}^{k},
\qquad k\in\{2,3\},
]

pero **cada técnica define de manera distinta qué información intenta conservar**.

Por eso no existe una única proyección "correcta": PCA preserva varianza; MDS distancias; Isomap distancias geodésicas; t-SNE vecindarios probabilísticos; LDA separación entre clases; UMAP relaciones topológicas/locales.
